# ModularEarth — Interactive Tutorial

This notebook mirrors `LayerTutorial.cpp` step by step using PyROOT (Python 3.12 + ROOT 6.36).

**Requirements before running:**
- Use `/usr/bin/python3.12` as the kernel
- Run from the project root: `/home/ydenizhernandez/ModularEarth/`
- `libModularEarth.so` must be built (`make` in the project root)

---
## 1 — Environment setup

In [2]:
import os, sys

# ── ROOT needs its lib dir on PYTHONPATH and LD_LIBRARY_PATH ──────────────────
ROOT_LIB = '/home/ydenizhernandez/root-install/lib'
os.environ['LD_LIBRARY_PATH'] = ROOT_LIB + ':' + os.environ.get('LD_LIBRARY_PATH', '')
if ROOT_LIB not in sys.path:
    sys.path.insert(0, ROOT_LIB)

# Work from the project root so relative .so paths resolve correctly
PROJECT_ROOT = '/home/ydenizhernandez/ModularEarth'
os.chdir(PROJECT_ROOT)

import ROOT
print('ROOT version:', ROOT.__version__)
print('Python:      ', sys.version)

print(sys.executable)
print(sys.version)

import ROOT
print(ROOT.gROOT.GetVersion())

ROOT version: 6.36.000
Python:       3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
/usr/bin/python3
3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
6.36.000


## 2 — Load the ModularEarth shared library

In [2]:
ret = ROOT.gSystem.Load('./libModularEarth.so')
if ret < 0:
    raise RuntimeError('Failed to load libModularEarth.so — run `make` first')
print('libModularEarth.so loaded (return code:', ret, ')')

ME = ROOT.ModularEarth   # namespace shortcut used throughout this notebook

libModularEarth.so loaded (return code: 0 )


Error in <TCling::LoadPCM>: ROOT PCM /home/ydenizhernandez/ModularEarth/ModularEarthDict_rdict.pcm file does not exist
Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libASImageGui_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libASImage_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libEG_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libEve_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libFITSIO_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libFitPanel_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizhernandez/root-install/lib/libFoam_rdict.pcm

Info in <TCling::LoadPCM>: In-memory ROOT PCM candidate /home/ydenizher

## 3 — Helper: build a `std::vector<SublayerConfig>` from Python tuples

Every `Layer.SetSubLayers()` call needs a `vector<SublayerConfig>`.  
Each row is `(sublayer_index, rmin_km, rmax_km, density_g_cm3, Z, A, material_name)`.

In [3]:
def make_sublayer_configs(rows):
    """Convert a list of tuples into a std::vector<ME.SublayerConfig>."""
    v = ROOT.std.vector[ME.SublayerConfig]()
    for (idx, rmin, rmax, rho, Z, A, name) in rows:
        cfg = ME.SublayerConfig(
            int(idx), float(rmin), float(rmax),
            float(rho), float(A), float(Z), str(name)
        )
        v.push_back(cfg)
    return v

print('Helper ready.')

Helper ready.


## 4 — Create the ROOT geometry manager and top volume

In [4]:
geoManager = ROOT.TGeoManager('TopVol', 'ModularEarth geometry')
matVac  = ROOT.TGeoMaterial('Vacuum', 0, 0, 0)
medVac  = ROOT.TGeoMedium('Vacuum', 1, matVac)
top     = ROOT.TGeoVolume('EarthTopSphere',
                           ROOT.TGeoSphere('Sphere', 0, 7000), medVac)
geoManager.SetTopVolume(top)
print('TGeoManager created, top sphere radius = 7000 km')

TGeoManager created, top sphere radius = 7000 km


Info in <TGeoManager::TGeoManager>: Geometry TopVol, ModularEarth geometry created
Info in <TGeoManager::SetTopVolume>: Top volume is EarthTopSphere. Master volume is EarthTopSphere
Info in <TGeoNavigator::BuildCache>: --- Maximum geometry depth set to 100


## 5 — Inner Core (solid sphere, bypasses Layer class)

Radius 0–1221.5 km.  Built directly as a `TGeoVolume` — same as the C++ tutorial.

In [5]:
matIC   = ROOT.TGeoMaterial('SolidNiFe', 57, 26, 12.98)
medIC   = ROOT.TGeoMedium('SolidNiFe', 2, matIC)
InnerCore = ROOT.TGeoVolume('InnerCore',
                             ROOT.TGeoSphere('InnerCoreSphere', 0, 1221.5), medIC)
InnerCore.SetLineColor(2)
print('InnerCore volume created (0–1221.5 km)')

InnerCore volume created (0–1221.5 km)


## 6 — Outer Core (Layer class, 5 sublayers)

In [6]:
dtheta, dphi = 20, 20   # angular bin sizes in degrees

OuterCore = ME.Layer('OuterCore', 2, dphi, dtheta)
oc_rows = [
    (1, 1221.5, 1946.7, 11.9512, 0.4691, 1, 'FeNi'),
    (2, 1946.7, 2475.1, 11.4726, 0.4691, 1, 'FeNi'),
    (3, 2475.1, 2900.3, 10.9884, 0.4691, 1, 'FeNi'),
    (4, 2900.3, 3260.6, 10.5008, 0.4691, 1, 'FeNi'),
    (5, 3260.6, 3480.0, 10.0861, 0.4691, 1, 'FeNi'),
]
OuterCore.SetSubLayers(make_sublayer_configs(oc_rows))
OuterCore.BuildLayer()
print('OuterCore built:', len(oc_rows), 'sublayers, 1221.5–3480 km')

OuterCore built: 5 sublayers, 1221.5–3480 km


Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material FeNi defined with fractional Z=0.469100
Warning in <TGeo

## 7 — Lower Mantle (3 sublayers)

In [7]:
LowerMantle = ME.Layer('LowerMantle', 3, dphi, dtheta)
lm_rows = [
    (1, 3480.0, 4311.0, 5.3243, 0.4691, 1, 'SilicateLM'),
    (2, 4311.0, 5073.6, 4.9315, 0.4691, 1, 'SilicateLM'),
    (3, 5073.6, 5701.0, 4.4916, 0.4691, 1, 'SilicateLM'),
]
LowerMantle.SetSubLayers(make_sublayer_configs(lm_rows))
LowerMantle.BuildLayer()
print('LowerMantle built:', len(lm_rows), 'sublayers, 3480–5701 km')

LowerMantle built: 3 sublayers, 3480–5701 km


Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>: Material SilicateLM defined with fractional Z=0.469100
Warning in <TGeoMaterial::ctor>:

## 8 — Upper Mantle with block-level material modification

This demonstrates heterogeneity injection: a small set of blocks in sublayer 2 is assigned an "AnomalousMantle" material (silicon-like), simulating e.g. a ULVZ patch.

In [8]:
UpperMantle = ME.Layer('UpperMantle', 4, dphi, dtheta)
um_rows = [
    (1, 5701.0, 5971.0, 3.8983, 0.4954, 1, 'SilicateUM'),
    (2, 5971.0, 6151.0, 3.6407, 0.4691, 1, 'SilicateUM'),
    (3, 6151.0, 6356.0, 3.3793, 0.4691, 1, 'SilicateUM'),
]
UpperMantle.SetSubLayers(make_sublayer_configs(um_rows))

# --- Modify a few blocks in sublayer 2 before building ---
target_blocks = [80, 81, 93]   # 1-based block IDs

blockIDs  = ROOT.std.vector['int']()
newA      = ROOT.std.vector['double']()
newZ      = ROOT.std.vector['double']()
newRho    = ROOT.std.vector['double']()
for bid in target_blocks:
    blockIDs.push_back(bid)
for a   in [28.0, 28.0, 28.0]: newA.push_back(a)
for z   in [14.0, 14.0, 14.0]: newZ.push_back(z)
for rho in [3.5,  3.6,  3.4]:  newRho.push_back(rho)

UpperMantle.UpdateBlocksInSubLayer(2, blockIDs, 'AnomalousMantle', newA, newZ, newRho)
print('Applied AnomalousMantle to blocks', target_blocks, 'in UpperMantle sublayer 2')

UpperMantle.SubLayerSummary(2)
UpperMantle.BuildLayer()
print('UpperMantle built.')

Applied AnomalousMantle to blocks [80, 81, 93] in UpperMantle sublayer 2
UpperMantle built.
Block 80 in Sublayer 2 updated with material: AnomalousMantle
Block 81 in Sublayer 2 updated with material: AnomalousMantle
Block 93 in Sublayer 2 updated with material: AnomalousMantle
=== Block Summary ===
  Block ID:        1
  Block Name:      BlockVolume_1
  Shape:           TGeoSphere
    rmin:          5971
    rmax:          6151
    thetamin:      0
    thetamax:      20
    phimin:        0
    phimax:        20
  Specific Block Modified: false
  Sub Layer Modified:      false
  Material:        SilicateUM
  Atomic Num  Z:   0.4691
  Atomic Mass A:   1
  Density:         3.6407 g/cm^3
  Pointers:
    Volume:        0x19ecfd60
    Material:      0x19ecf890
    Medium:        0x19ecfae0
=== Block Summary ===
  Block ID:        2
  Block Name:      BlockVolume_2
  Shape:           TGeoSphere
    rmin:          5971
    rmax:          6151
    thetamin:      0
    thetamax:      20
    phi

Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>: Material SilicateUM defined with fractional Z=0.495400
Warning in <TGeoMaterial::ctor>:

## 9 — Crust, Ocean, Atmosphere

In [9]:
# Crust
Crust = ME.Layer('Crust', 5, dphi, dtheta)
Crust.SetSubLayers(make_sublayer_configs([
    (1, 6356.0, 6368.0, 2.9, 0.4957, 1, 'CrustSi'),
]))
Crust.BuildLayer()

# Ocean
Ocean = ME.Layer('Ocean', 6, dphi, dtheta)
Ocean.SetSubLayers(make_sublayer_configs([
    (1, 6368.0, 6371.0, 1.02, 0.5551, 1, 'Water'),
]))
Ocean.BuildLayer()

# Atmosphere (very low density)
Atmosphere = ME.Layer('Atmosphere', 7, dphi, dtheta)
Atmosphere.SetSubLayers(make_sublayer_configs([
    (1, 6371.0, 6386.0, 0.001, 0.4991, 1, 'Vac'),
]))
Atmosphere.BuildLayer()

print('Crust, Ocean, Atmosphere built.')

Crust, Ocean, Atmosphere built.


Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fractional Z=0.495700
Warning in <TGeoMaterial::ctor>: Material CrustSi defined with fr

## 10 — Assemble all layers into the top volume

In [10]:
origin = ROOT.TGeoTranslation(0, 0, 0)

top.AddNode(InnerCore,                    1, origin)
top.AddNode(OuterCore.GetLayerVolume(),   1, origin)
top.AddNode(LowerMantle.GetLayerVolume(), 1, origin)
top.AddNode(UpperMantle.GetLayerVolume(), 1, origin)
top.AddNode(Crust.GetLayerVolume(),       1, origin)
top.AddNode(Ocean.GetLayerVolume(),       1, origin)
top.AddNode(Atmosphere.GetLayerVolume(),  1, origin)

print('All', top.GetNdaughters(), 'layer volumes added to top.')

All 7 layer volumes added to top.


## 11 — Export block-vertex CSV maps

Writes CSV files with `(block_id, mod_flag, phi1, phi2, theta1, theta2, x, y, z)` per block — used by `mollwide2.py` for Mollweide map visualisation.

In [11]:
LowerMantle.SubLayerBlockMap(1)
UpperMantle.SubLayerBlockMap(2)

import glob
csvs = glob.glob('*BlockMap.csv')
print('CSV files written:', csvs)

CSV files written: ['LowerMantle_3_1-SubLayer_1_1_BlockMap.csv', 'UpperMantle_4_1-SubLayer_2_2_BlockMap.csv']
Total number of layers in TOP (# OF layers): 7
This Layer's LayerID: 3
Total number of SubLayers mother Layer: 3
SUBLAYER node name [1]: SubLayer_1_1
SUBLAYER Path [1]: /EarthTopSphere_1/LowerMantle_3_1/SubLayer_1_1
SUBLAYER NODE: /EarthTopSphere_1/LowerMantle_3_1/SubLayer_1_1
  BLOCKS IN THIS SUBLAYER: 162
  BLOCK [0]: modFlag=0
  BLOCK [0]: /EarthTopSphere_1/LowerMantle_3_1/SubLayer_1_1/BlockVolume_1_1
  Angular coordinates limits: (0.0000, 20.0000,0.0000, 20.0000)
  Radial coordinates limits: (3480.0000, 4311.0000)
  World: (0.0000, 0.0000, 4311.0000)
  World: (1474.4488, 0.0000, 4051.0149)
  World: (0.0000, 0.0000, 4311.0000)
  World: (1385.5287, 504.2912, 4051.0149)
  BLOCK [1]: modFlag=0
  BLOCK [1]: /EarthTopSphere_1/LowerMantle_3_1/SubLayer_1_1/BlockVolume_2_2
  Angular coordinates limits: (20.0000, 40.0000,0.0000, 20.0000)
  Radial coordinates limits: (3480.0000, 4311.

## 12 — Inspect geometry and print summary

> **Known limitation — `CloseGeometry()` segfaults from Python:**  
> Calling `geoManager.CloseGeometry()` invokes `TGeoManager::CheckGeometry()` internally, which segfaults when called via PyROOT on assembly volumes in this ROOT 6.36 build.  
> The **C++ tutorial** (`make LayerTutorial`) runs `CloseGeometry()` without issue via ROOT's cling interpreter.  
> For notebook work (summaries, CSV exports, material queries) you do **not** need to close the geometry.

In [12]:
# Roll call: prints all registered layer/sublayer names and IDs in gGeoManager
LowerMantle.Call_The_Roll()

# Node count (no CloseGeometry needed for this)
print('Top volume daughters:', top.GetNdaughters())

Top volume daughters: 7
Calling the roll for current model setup: Method from Layer LowerMantle_3 ===

NODE IN TOP [0]: InnerCore_1
NODE IN TOP [1]: OuterCore_2_1
Total number of SubLayers in Layer 1: 5
NODE IN TOP [2]: LowerMantle_3_1
Total number of SubLayers in Layer 2: 3
NODE IN TOP [3]: UpperMantle_4_1
Total number of SubLayers in Layer 3: 3
NODE IN TOP [4]: Crust_5_1
Total number of SubLayers in Layer 4: 1
NODE IN TOP [5]: Ocean_6_1
Total number of SubLayers in Layer 5: 1
NODE IN TOP [6]: Atmosphere_7_1
Total number of SubLayers in Layer 6: 1


## 13 — (Optional) Visualise block map with matplotlib

Reads the exported CSV and plots blocks colour-coded by modification flag on a Mollweide projection (requires `matplotlib` and `basemap` or `cartopy`).

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

csv_file = 'UpperMantle_4_1-SubLayer_2_2_BlockMap.csv'
df = pd.read_csv(csv_file)
print(df.head())
print('Columns:', list(df.columns))

   block_id  mod_flag  phi1  phi2  theta1  theta2          x         y  \
0         0         0   0.0  20.0     0.0    20.0     0.0000    0.0000   
1         0         0   0.0  20.0     0.0    20.0  2103.7659    0.0000   
2         0         0   0.0  20.0     0.0    20.0     0.0000    0.0000   
3         0         0   0.0  20.0     0.0    20.0  1976.8933  719.5303   
4         1         0  20.0  40.0     0.0    20.0     0.0000    0.0000   

           z  
0  6151.0000  
1  5780.0493  
2  6151.0000  
3  5780.0493  
4  6151.0000  
Columns: ['block_id', 'mod_flag', 'phi1', 'phi2', 'theta1', 'theta2', 'x', 'y', 'z']


---
## What's working vs. what's not yet implemented

| Feature | Status |
|---|---|
| `Block` — geometry + material | ✅ Complete |
| `Sublayer` — assembly + per-block material update | ✅ Complete |
| `Layer` — multi-sublayer + CSV export | ✅ Complete |
| `Earth.SetEarthModel()` (PREM table reader) | ✅ Complete |
| `Earth.BuildEarth()` | ❌ Not yet ported from `Earth.hxx` |
| `Earth.GetEarthPath()` (neutrino track) | ❌ Not yet ported |
| `Earth.SetDetectorLocation()` | ❌ Not yet ported |
| PyROOT / Jupyter | ✅ Works with Python 3.12 (`/usr/bin/python3.12`) |

**Python kernel requirement:** this notebook must run on the **Python 3.12** kernel.  
The pyenv-managed Python 3.7.5 in this repo will fail with `undefined symbol: PyErr_GetRaisedException` because ROOT 6.36 was compiled against Python 3.12's C ABI.